In [41]:
import tkinter as tk
import requests
import os
from dotenv import load_dotenv
import google.generativeai as genai

# Tải khóa API từ file .env
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GOOGLE_SEARCH_API_KEY = os.getenv("GOOGLE_SEARCH_API_KEY")
GOOGLE_SEARCH_CX = os.getenv("GOOGLE_SEARCH_CX")

# Kiểm tra và xác thực các khóa API
if not all([GEMINI_API_KEY, GOOGLE_SEARCH_API_KEY, GOOGLE_SEARCH_CX]):
    raise ValueError("⚠️ Thiếu khóa API trong file .env!")

# Cấu hình API Gemini
genai.configure(api_key=GEMINI_API_KEY)

class TacNhan:
    def __init__(self, ten):
        """
        Khởi tạo một tác nhân với tên được chỉ định
        
        :param ten: Tên của tác nhân
        """
        self.ten = ten

    def xu_ly(self, yeu_cau_nguoi_dung):
        """
        Xử lý yêu cầu của người dùng dựa trên loại tác nhân
        
        :param yeu_cau_nguoi_dung: Câu hỏi hoặc yêu cầu của người dùng
        :return: Kết quả xử lý từ tác nhân
        """
        try:
            if self.ten == "Gemini":
                return self.hoi_gemini(yeu_cau_nguoi_dung)
            elif self.ten == "TimKiem":
                return self.tim_kiem_google(yeu_cau_nguoi_dung)
            else:
                return f"Tác nhân {self.ten}: Không có phương thức xử lý."
        except Exception as e:
            return f"Tác nhân {self.ten}: Lỗi khi xử lý yêu cầu. Chi tiết: {str(e)}"

    def hoi_gemini(self, van_ban):
        """
        Gọi API Gemini để trả lời câu hỏi
        
        :param van_ban: Văn bản câu hỏi
        :return: Phản hồi từ Gemini
        """
        try:
            # Chức năng để in ra danh sách model
            def in_danh_sach_model():
                print("Danh sách model Gemini:")
                models = genai.list_models()
                for model in models:
                    try:
                        print(f"Model: {model.name}")
                    except Exception as e:
                        print(f"Lỗi khi in model: {e}")

            # In danh sách model
            in_danh_sach_model()
            
            # Thử nghiệm với các model phổ biến
            cac_model_thu = [
                'gemini-1.5-pro-latest', 
                'gemini-pro', 
                'gemini-1.0-pro',
                'gemini-1.5-flash-latest'
            ]
            
            # Thử từng model
            for model_name in cac_model_thu:
                try:
                    # Tạo model
                    model = genai.GenerativeModel(model_name)
                    
                    # Cấu hình sinh văn bản
                    cau_hinh_sinh = {
                        "temperature": 0.7,  # Độ sáng tạo của câu trả lời
                        "max_output_tokens": 2048,  # Số token tối đa cho câu trả lời
                    }
                    
                    # Sinh nội dung
                    phan_hoi = model.generate_content(
                        van_ban, 
                        generation_config=cau_hinh_sinh
                    )
                    
                    return f"Tác nhân Gemini: {phan_hoi.text.strip()}"
                
                except Exception as model_error:
                    print(f"Lỗi với model {model_name}: {model_error}")
            
            # Nếu không model nào hoạt động
            raise ValueError("Không tìm thấy model Gemini phù hợp")
        
        except Exception as e:
            return f"Tác nhân Gemini: Lỗi gọi API. Chi tiết: {str(e)}"

    def tim_kiem_google(self, truy_van):
        """
        Gửi yêu cầu tìm kiếm đến Google Custom Search API
        
        :param truy_van: Từ khóa tìm kiếm
        :return: Kết quả tìm kiếm
        """
        try:
            url = "https://www.googleapis.com/customsearch/v1"
            tham_so = {
                "q": truy_van,
                "key": GOOGLE_SEARCH_API_KEY,
                "cx": GOOGLE_SEARCH_CX
            }
            phan_hoi = requests.get(url, params=tham_so, timeout=10)
            
            if phan_hoi.status_code == 200:
                du_lieu = phan_hoi.json()
                if 'items' in du_lieu:
                    ket_qua = du_lieu['items'][0]['snippet']
                    return f"Tác nhân Tìm kiếm: Kết quả tìm kiếm: {ket_qua}"
                else:
                    return "Tác nhân Tìm kiếm: Không tìm thấy kết quả."
            else:
                return f"Tác nhân Tìm kiếm: Lỗi tìm kiếm. Mã trạng thái: {phan_hoi.status_code}"
        except requests.RequestException as e:
            return f"Tác nhân Tìm kiếm: Lỗi mạng. Chi tiết: {str(e)}"

class DieuPhoi:
    def __init__(self):
        """
        Khởi tạo hệ thống điều phối các tác nhân
        """
        self.cac_tac_nhan = []

    def them_tac_nhan(self, tac_nhan):
        """
        Thêm tác nhân vào hệ thống
        
        :param tac_nhan: Tác nhân cần thêm
        """
        self.cac_tac_nhan.append(tac_nhan)

    def phan_phoi_nhiem_vu(self, yeu_cau_nguoi_dung):
        """
        Phân phối nhiệm vụ đến tất cả các tác nhân và thu thập kết quả
        
        :param yeu_cau_nguoi_dung: Yêu cầu của người dùng
        :return: Danh sách kết quả từ các tác nhân
        """
        ket_qua = []
        for tac_nhan in self.cac_tac_nhan:
            ket_qua_tac_nhan = tac_nhan.xu_ly(yeu_cau_nguoi_dung)
            ket_qua.append((tac_nhan.ten, ket_qua_tac_nhan))
        return ket_qua

class GiaoDienNguoiDung:
    def __init__(self, man_hinh_chinh):
        """
        Khởi tạo giao diện người dùng
        
        :param man_hinh_chinh: Cửa sổ giao diện chính
        """
        self.man_hinh_chinh = man_hinh_chinh
        man_hinh_chinh.title("Hệ Thống Tác Nhân Thông Minh")
        man_hinh_chinh.geometry("500x600")

        # Tạo hệ thống điều phối và thêm tác nhân
        self.dieu_phoi = DieuPhoi()
        self.dieu_phoi.them_tac_nhan(TacNhan("Gemini"))
        self.dieu_phoi.them_tac_nhan(TacNhan("TimKiem"))

        # Tạo và thiết lập các thành phần giao diện
        self.tao_cac_phan_tu()

    def tao_cac_phan_tu(self):
        """
        Tạo và bố trí các phần tử giao diện
        """
        # Nhãn hướng dẫn nhập câu hỏi
        tk.Label(self.man_hinh_chinh, text="Nhập câu hỏi của bạn:").pack(pady=10)

        # Ô nhập câu hỏi
        self.o_nhap = tk.Entry(self.man_hinh_chinh, width=60)
        self.o_nhap.pack(pady=10)
        self.o_nhap.bind('<Return>', self.hoi_cau_hoi)  # Hỗ trợ phím Enter

        # Nút "Hỏi"
        nut_hoi = tk.Button(self.man_hinh_chinh, text="Hỏi", command=self.hoi_cau_hoi)
        nut_hoi.pack(pady=10)

        # Vùng hiển thị kết quả
        self.vung_ket_qua = tk.Text(self.man_hinh_chinh, height=20, width=60, wrap=tk.WORD)
        self.vung_ket_qua.pack(pady=10)

        # Thanh cuộn cho vùng kết quả
        thanh_cuon = tk.Scrollbar(self.man_hinh_chinh, command=self.vung_ket_qua.yview)
        thanh_cuon.pack(side=tk.RIGHT, fill=tk.Y)
        self.vung_ket_qua.config(yscrollcommand=thanh_cuon.set)

    def hoi_cau_hoi(self, su_kien=None):
        """
        Xử lý câu hỏi của người dùng và hiển thị kết quả
        
        :param su_kien: Sự kiện (được sử dụng khi nhấn Enter)
        """
        cau_hoi = self.o_nhap.get().strip()
        if cau_hoi:
            # Xóa kết quả cũ
            self.vung_ket_qua.delete(1.0, tk.END)
            
            # Phân phối nhiệm vụ và hiển thị kết quả
            ket_qua = self.dieu_phoi.phan_phoi_nhiem_vu(cau_hoi)
            for ten_tac_nhan, noi_dung in ket_qua:
                self.vung_ket_qua.insert(tk.END, f"{noi_dung}\n\n")
            
            # Xóa ô nhập
            self.o_nhap.delete(0, tk.END)

def chinh(danh_sach_tham_so=None):
    """
    Hàm chính để khởi chạy ứng dụng
    """
    man_hinh = tk.Tk()
    ung_dung = GiaoDienNguoiDung(man_hinh)
    man_hinh.mainloop()

if __name__ == "__main__":
    chinh()